In [ ]:
# Install dependencies
!pip install -q transformers accelerate "bitsandbytes>=0.46.1" pillow torch torchvision tqdm sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 47.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

# Define the model ID
model_id = "llava-hf/llava-1.5-7b-hf"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


processor = LlavaProcessor.from_pretrained(model_id)


model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affi

In [ ]:
import os

# Unzip MME subset
if not os.path.exists("/content/MME"):
    !unzip -q /content/MME.zip -d /content/MME

# Unzip POPE subset
if not os.path.exists("/content/POPE"):
    !unzip -q /content/POPE.zip -d /content/POPE

# Unzip MMVP subset
if not os.path.exists("/content/MMVP"):
    !unzip -q /content/MMVP.zip -d /content/MMVP

In [ ]:
# Var C
import json
import time
from PIL import Image
from tqdm import tqdm
import torch.nn.functional as F

def generate_benchmark_answers_variant_c(benchmark_dir, question_filename, output_filename, model_id_name="llava-v1.5-7b", drop_k=50):
    input_jsonl_path = os.path.join(benchmark_dir, "Questions", question_filename)
    output_jsonl_path = os.path.join(benchmark_dir, output_filename)
    images_dir = os.path.join(benchmark_dir, "Images")

    if not os.path.exists(input_jsonl_path):
        print(f"Warning: {input_jsonl_path} not found. Skipping...")
        return

    data = []
    with open(input_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))

    print(f"Loaded {len(data)} questions from {input_jsonl_path}")



    IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<image>")
    if IMAGE_TOKEN_ID is None:
        IMAGE_TOKEN_ID = processor.tokenizer.pad_token_id

    def get_token_ids(word_list):
        raw_ids = processor.tokenizer(word_list, add_special_tokens=False).input_ids
        return set(item for sublist in raw_ids for item in (sublist if isinstance(sublist, list) else [sublist]))

    yes_ids = get_token_ids(["Yes", " Yes", "yes", " yes"])
    no_ids = get_token_ids(["No", " No", "no", " no"])
    a_ids = get_token_ids(["(a)", " (a)", "(a", " (", "(", "a", " a", "A", " A"])
    b_ids = get_token_ids(["(b)", " (b)", "(b", "b", " b", "B", " B"])

    NUM_IMAGE_TOKENS = 576
    MAX_NEW_TOKENS = 50

    with open(output_jsonl_path, 'w', encoding='utf-8') as outfile:
        for item in tqdm(data, desc=f"Processing {os.path.basename(benchmark_dir)}"):
            image_filename = item.get("image", "")
            prompt_text = item.get("text", item.get("question", "")).strip()
            if not prompt_text:
                continue

            img_path = os.path.join(images_dir, image_filename)
            if not os.path.exists(img_path):
                img_path = os.path.join(benchmark_dir, image_filename)

            try:
                raw_image = Image.open(img_path).convert("RGB")
            except Exception as e:
                continue

            full_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"

            inputs = processor(text=full_prompt, images=raw_image, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.inference_mode():

                # Timer

                start_time = time.perf_counter()


                # Discovery

                outputs_base = model(**inputs, use_cache=True, output_hidden_states=False)
                past_key_values = outputs_base.past_key_values

                v_caches = []
                cache_iterable = past_key_values.to_legacy_cache() if hasattr(past_key_values, "to_legacy_cache") else (
                    list(zip(past_key_values.key_cache, past_key_values.value_cache)) if hasattr(past_key_values, "key_cache") else past_key_values)

                for layer_data in cache_iterable:
                    v_caches.append(layer_data[1])

                input_ids_list = inputs["input_ids"][0].tolist()
                image_token_index = 32000
                vis_start = input_ids_list.index(image_token_index) if image_token_index in input_ids_list else (
                    input_ids_list.index(IMAGE_TOKEN_ID) if IMAGE_TOKEN_ID in input_ids_list else -1)

                global_sink_indices = []
                if vis_start != -1:
                    vis_end = vis_start + NUM_IMAGE_TOKENS
                    v_norms = []
                    for v_cache in v_caches:
                        vis_v_cache = v_cache[0, :, vis_start:vis_end, :]
                        layer_v_norm = torch.linalg.norm(vis_v_cache.float(), dim=-1).mean(dim=0)
                        v_norms.append(layer_v_norm)

                    avg_v_norm = torch.stack(v_norms).mean(dim=0)
                    _, topk_indices = torch.topk(avg_v_norm, k=drop_k)
                    global_sink_indices = (topk_indices + vis_start).tolist()


                # Modify Mask and Pass

                modified_mask = inputs["attention_mask"].clone()
                if vis_start != -1:
                    modified_mask[0, global_sink_indices] = 0

                inputs["attention_mask"] = modified_mask


                outputs_var = model(**inputs, use_cache=True, output_hidden_states=False)

                first_token_logits = outputs_var.logits[0, -1, :]
                current_past_key_values = outputs_var.past_key_values


                # Prob. Logic

                probs = F.softmax(first_token_logits.float(), dim=-1)

                is_multiple_choice = "(a)" in prompt_text.lower() and "(b)" in prompt_text.lower()

                if is_multiple_choice:
                    opt1_prob = sum(probs[idx].item() for idx in a_ids if idx < probs.size(-1))
                    opt2_prob = sum(probs[idx].item() for idx in b_ids if idx < probs.size(-1))
                else:
                    opt1_prob = sum(probs[idx].item() for idx in yes_ids if idx < probs.size(-1))
                    opt2_prob = sum(probs[idx].item() for idx in no_ids if idx < probs.size(-1))

                top_k_probs, top_k_ids = torch.topk(probs, k=5)
                top_k_tokens = processor.tokenizer.convert_ids_to_tokens(top_k_ids)
                top_k_dict = {token: round(prob.item(), 4) for token, prob in zip(top_k_tokens, top_k_probs)}
                entropy = -(probs * torch.log(probs + 1e-9)).sum().item()


                # Manual Decoding Loop

                next_token_id = first_token_logits.argmax(dim=-1).unsqueeze(0).unsqueeze(0)
                generated_ids = [next_token_id.item()]
                current_input_ids = next_token_id

                current_attention_mask = modified_mask

                for _ in range(MAX_NEW_TOKENS - 1):
                    current_attention_mask = torch.cat(
                        [current_attention_mask, torch.ones((1, 1), device=model.device, dtype=torch.long)], dim=1
                    )

                    outputs = model(
                        input_ids=current_input_ids,
                        attention_mask=current_attention_mask,
                        past_key_values=current_past_key_values,
                        use_cache=True
                    )
                    current_past_key_values = outputs.past_key_values
                    next_token_id = outputs.logits[0, -1, :].argmax(dim=-1).unsqueeze(0).unsqueeze(0)

                    token_val = next_token_id.item()
                    generated_ids.append(token_val)
                    if token_val == processor.tokenizer.eos_token_id:
                        break

                    current_input_ids = next_token_id


                # END TIMER

                end_time = time.perf_counter()
                inference_time = end_time - start_time

            final_output_ids = torch.tensor([generated_ids], device=model.device)
            response_text = processor.decode(final_output_ids[0], skip_special_tokens=True).strip()

            output_item = {
                "question_id": item.get("question_id", ""),
                "prompt": prompt_text,
                "label": item.get("label", ""),
                "response": response_text,
                "image": image_filename,
                "model_id": f"{model_id_name}_varC",
                "opt1_prob": round(opt1_prob, 4),
                "opt2_prob": round(opt2_prob, 4),
                "entropy": round(entropy, 4),
                "top_5_distribution": top_k_dict,
                "inference_time": round(inference_time, 4)
            }
            outfile.write(json.dumps(output_item) + "\n")



In [ ]:
benchmarks = [
    {
        "name": "MME",
        "path": "/content/MME",
        "questions_file": "mme-questions.jsonl"
    },
    {
        "name": "MMVP",
        "path": "/content/MMVP",
        "questions_file": "mmvp-questions.jsonl"
    },
    {
        "name": "POPE",
        "path": "/content/POPE",
        "questions_file": "pope-questions.jsonl"
    }
]

for bm in benchmarks:


    output_filename = f"{bm['name'].lower()}_llava_v1_5_7b_answers_c.jsonl"

    generate_benchmark_answers_variant_c(
        benchmark_dir=bm["path"],
        question_filename=bm["questions_file"],
        output_filename=output_filename,
        model_id_name="llava-v1.5-7b"
    )



Loaded 200 questions from /content/MME/Questions/mme-questions.jsonl


Processing MME: 100%|██████████| 200/200 [02:23<00:00,  1.39it/s]


Loaded 300 questions from /content/MMVP/Questions/mmvp-questions.jsonl


Processing MMVP: 100%|██████████| 300/300 [05:46<00:00,  1.15s/it]


Loaded 80 questions from /content/POPE/Questions/pope-questions.jsonl


Processing POPE:  44%|████▍     | 35/80 [01:21<01:42,  2.28s/it]

In [ ]:
# Evaluate MME Results
from collections import defaultdict

result_file = "/content/MME/mme_llava_v1_5_7b_answers_c.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the MME jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"Evaluating MME Benchmark for: {result_file}\n")
    print("-" * 50)


    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    results_by_category = defaultdict(lambda: defaultdict(list))

    for item in answers:
        gt = str(item.get('label', '')).lower().strip()
        text = str(item.get('response', '')).lower().strip()


        if '.' in text:
            text = text.split('.')[0]
        text = text.replace(',', '').split(' ')

        if 'no' in text or 'not' in text:
            pred = 'no'
        else:
            pred = 'yes'

        is_correct = (pred == gt)


        question_id = item.get("question_id", "")
        if "_" in question_id:

            img_id = question_id.rsplit('_', 1)[0]
        else:
            img_id = question_id


        image_path = item.get('image', 'unknown.jpg')
        if 'category' in item:
            category = item['category']
        elif '/' in image_path:
            category = image_path.split('/')[-2]
        else:
            category = 'Overall'

        results_by_category[category][img_id].append(is_correct)

    total_acc_sum = 0
    total_acc_plus_sum = 0

    print(f"{'Category':<20} | {'Accuracy':<10} | {'Accuracy+':<10} | {'Score (Max 200)':<10}")
    print("-" * 60)

    for category, images in results_by_category.items():
        total_questions = 0
        correct_questions = 0

        total_images = 0
        correct_images = 0

        for img_id, outcomes in images.items():
            total_images += 1
            total_questions += len(outcomes)
            correct_questions += sum(outcomes)


            if len(outcomes) == 2 and sum(outcomes) == 2:
                correct_images += 1

        acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
        acc_plus = (correct_images / total_images) * 100 if total_images > 0 else 0
        score = acc + acc_plus

        total_acc_sum += acc
        total_acc_plus_sum += acc_plus

        print(f"{category:<20} | {acc:<10.2f} | {acc_plus:<10.2f} | {score:<10.2f}")

    print("-" * 60)

    num_categories = len(results_by_category)
    if num_categories > 0:
        avg_acc = total_acc_sum / num_categories
        avg_acc_plus = total_acc_plus_sum / num_categories
        total_score = total_acc_sum + total_acc_plus_sum
        print(f"{'TOTAL / AVERAGE':<20} | {avg_acc:<10.2f} | {avg_acc_plus:<10.2f} | {total_score:<10.2f}")

Evaluating MME Benchmark for: /content/MME/mme_llava_v1_5_7b_answers_c.jsonl

--------------------------------------------------
Category             | Accuracy   | Accuracy+  | Score (Max 200)
------------------------------------------------------------
Overall              | 73.00      | 47.00      | 120.00    
------------------------------------------------------------
TOTAL / AVERAGE      | 73.00      | 47.00      | 120.00    


In [ ]:
# Evaluate POPE Results

result_file = "/content/POPE/pope_llava_v1_5_7b_answers_c.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the POPE jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"✅ Found file! Evaluating: {result_file}\n")
    print("-" * 50)

    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    pred_list = []
    label_list = []

    for item in answers:

        gt = str(item.get('label', '')).lower().strip()
        label_list.append(0 if gt == 'no' else 1)


        text = str(item.get('response', '')).strip()


        if '.' in text:
            text = text.split('.')[0]

        text = text.replace(',', '')
        words = text.split(' ')


        if 'No' in words or 'not' in words or 'no' in words:
            pred_list.append(0)
        else:
            pred_list.append(1)

    pos, neg = 1, 0
    yes_ratio = pred_list.count(1) / len(pred_list) if pred_list else 0

    TP, TN, FP, FN = 0, 0, 0, 0
    for pred, label in zip(pred_list, label_list):
        if pred == pos and label == pos:
            TP += 1
        elif pred == pos and label == neg:
            FP += 1
        elif pred == neg and label == neg:
            TN += 1
        elif pred == neg and label == pos:
            FN += 1

    print('TP\tFP\tTN\tFN')
    print(f'{TP}\t{FP}\t{TN}\t{FN}\n')

    precision = float(TP) / float(TP + FP) if (TP + FP) > 0 else 0
    recall = float(TP) / float(TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1 score:  {f1:.4f}')
    print(f'Yes ratio: {yes_ratio:.4f}')
    print("-" * 50)

✅ Found file! Evaluating: /content/POPE/pope_llava_v1_5_7b_answers_c.jsonl

--------------------------------------------------
TP	FP	TN	FN
34	6	34	6

Accuracy:  0.8500
Precision: 0.8500
Recall:    0.8500
F1 score:  0.8500
Yes ratio: 0.5000
--------------------------------------------------


In [ ]:
# Var A


def generate_benchmark_answers_variant_a(benchmark_dir, question_filename, output_filename, model_id_name="llava-v1.5-7b", num_static_sinks=4):
    input_jsonl_path = os.path.join(benchmark_dir, "Questions", question_filename)
    output_jsonl_path = os.path.join(benchmark_dir, output_filename)
    images_dir = os.path.join(benchmark_dir, "Images")

    if not os.path.exists(input_jsonl_path):
        print(f"Warning: {input_jsonl_path} not found. Skipping...")
        return

    data = []
    with open(input_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))

    print(f"Loaded {len(data)} questions from {input_jsonl_path}")



    IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<image>")
    if IMAGE_TOKEN_ID is None:
        IMAGE_TOKEN_ID = processor.tokenizer.pad_token_id

    def get_token_ids(word_list):
        raw_ids = processor.tokenizer(word_list, add_special_tokens=False).input_ids
        return set(item for sublist in raw_ids for item in (sublist if isinstance(sublist, list) else [sublist]))

    yes_ids = get_token_ids(["Yes", " Yes", "yes", " yes"])
    no_ids = get_token_ids(["No", " No", "no", " no"])
    a_ids = get_token_ids(["(a)", " (a)", "(a", " (", "(", "a", " a", "A", " A"])
    b_ids = get_token_ids(["(b)", " (b)", "(b", "b", " b", "B", " B"])

    NUM_IMAGE_TOKENS = 576
    MAX_NEW_TOKENS = 50

    with open(output_jsonl_path, 'w', encoding='utf-8') as outfile:
        for item in tqdm(data, desc=f"Processing {os.path.basename(benchmark_dir)}"):
            image_filename = item.get("image", "")
            prompt_text = item.get("text", item.get("question", "")).strip()
            if not prompt_text:
                continue

            img_path = os.path.join(images_dir, image_filename)
            if not os.path.exists(img_path):
                img_path = os.path.join(benchmark_dir, image_filename)

            try:
                raw_image = Image.open(img_path).convert("RGB")
            except Exception as e:
                continue

            full_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"

            inputs = processor(text=full_prompt, images=raw_image, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.inference_mode():

                # Static Bias

                input_ids_list = inputs["input_ids"][0].tolist()
                image_token_index = 32000
                vis_start = input_ids_list.index(image_token_index) if image_token_index in input_ids_list else (
                    input_ids_list.index(IMAGE_TOKEN_ID) if IMAGE_TOKEN_ID in input_ids_list else -1)

                modified_mask = inputs["attention_mask"].clone()

                if vis_start != -1:
                    vis_end = vis_start + NUM_IMAGE_TOKENS
                    static_sink_indices = list(range(vis_start, vis_start + num_static_sinks)) + \
                                          list(range(vis_end - num_static_sinks, vis_end))
                    modified_mask[0, static_sink_indices] = 0

                inputs["attention_mask"] = modified_mask


                # Time

                start_time = time.perf_counter()

                outputs = model(**inputs, use_cache=True, output_hidden_states=False)

                first_token_logits = outputs.logits[0, -1, :]
                current_past_key_values = outputs.past_key_values

                # Prob Logic

                probs = F.softmax(first_token_logits.float(), dim=-1)

                is_multiple_choice = "(a)" in prompt_text.lower() and "(b)" in prompt_text.lower()

                if is_multiple_choice:
                    opt1_prob = sum(probs[idx].item() for idx in a_ids if idx < probs.size(-1))
                    opt2_prob = sum(probs[idx].item() for idx in b_ids if idx < probs.size(-1))
                else:
                    opt1_prob = sum(probs[idx].item() for idx in yes_ids if idx < probs.size(-1))
                    opt2_prob = sum(probs[idx].item() for idx in no_ids if idx < probs.size(-1))

                top_k_probs, top_k_ids = torch.topk(probs, k=5)
                top_k_tokens = processor.tokenizer.convert_ids_to_tokens(top_k_ids)
                top_k_dict = {token: round(prob.item(), 4) for token, prob in zip(top_k_tokens, top_k_probs)}
                entropy = -(probs * torch.log(probs + 1e-9)).sum().item()




                next_token_id = first_token_logits.argmax(dim=-1).unsqueeze(0).unsqueeze(0)
                generated_ids = [next_token_id.item()]
                current_input_ids = next_token_id

                current_attention_mask = modified_mask

                for _ in range(MAX_NEW_TOKENS - 1):
                    current_attention_mask = torch.cat(
                        [current_attention_mask, torch.ones((1, 1), device=model.device, dtype=torch.long)], dim=1
                    )

                    outputs = model(
                        input_ids=current_input_ids,
                        attention_mask=current_attention_mask,
                        past_key_values=current_past_key_values,
                        use_cache=True
                    )
                    current_past_key_values = outputs.past_key_values
                    next_token_id = outputs.logits[0, -1, :].argmax(dim=-1).unsqueeze(0).unsqueeze(0)

                    token_val = next_token_id.item()
                    generated_ids.append(token_val)
                    if token_val == processor.tokenizer.eos_token_id:
                        break

                    current_input_ids = next_token_id



                end_time = time.perf_counter()
                inference_time = end_time - start_time


            final_output_ids = torch.tensor([generated_ids], device=model.device)
            response_text = processor.decode(final_output_ids[0], skip_special_tokens=True).strip()

            output_item = {
                "question_id": item.get("question_id", ""),
                "prompt": prompt_text,
                "label": item.get("label", ""),
                "response": response_text,
                "image": image_filename,
                "model_id": f"{model_id_name}_varA",
                "opt1_prob": round(opt1_prob, 4),
                "opt2_prob": round(opt2_prob, 4),
                "entropy": round(entropy, 4),
                "top_5_distribution": top_k_dict,
                "inference_time": round(inference_time, 4)
            }
            outfile.write(json.dumps(output_item) + "\n")



In [ ]:
benchmarks = [
    {
        "name": "MME",
        "path": "/content/MME",
        "questions_file": "mme-questions.jsonl"
    },
    {
        "name": "MMVP",
        "path": "/content/MMVP",
        "questions_file": "mmvp-questions.jsonl"
    },
    {
        "name": "POPE",
        "path": "/content/POPE",
        "questions_file": "pope-questions.jsonl"
    }
]

for bm in benchmarks:
    print(f"\n==========================================")
    print(f"Starting inference for {bm['name']}...")

    output_filename = f"{bm['name'].lower()}_llava_v1_5_7b_answers_var_a.jsonl"

    generate_benchmark_answers_variant_a(
        benchmark_dir=bm["path"],
        question_filename=bm["questions_file"],
        output_filename=output_filename,
        model_id_name="llava-v1.5-7b"
    )




Starting inference for MME...
Loaded 200 questions from /content/MME/Questions/mme-questions.jsonl


Processing MME: 100%|██████████| 200/200 [01:22<00:00,  2.43it/s]



Starting inference for MMVP...
Loaded 300 questions from /content/MMVP/Questions/mmvp-questions.jsonl


Processing MMVP: 100%|██████████| 300/300 [04:17<00:00,  1.17it/s]



Starting inference for POPE...
Loaded 80 questions from /content/POPE/Questions/pope-questions.jsonl


Processing POPE: 100%|██████████| 80/80 [02:35<00:00,  1.95s/it]


In [ ]:
# Evaluate MME Results

result_file = "/content/MME/mme_llava_v1_5_7b_answers_var_a.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the MME jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"Evaluating MME Benchmark for: {result_file}\n")
    print("-" * 50)


    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    results_by_category = defaultdict(lambda: defaultdict(list))

    for item in answers:
        gt = str(item.get('label', '')).lower().strip()
        text = str(item.get('response', '')).lower().strip()


        if '.' in text:
            text = text.split('.')[0]
        text = text.replace(',', '').split(' ')

        if 'no' in text or 'not' in text:
            pred = 'no'
        else:
            pred = 'yes'

        is_correct = (pred == gt)


        question_id = item.get("question_id", "")
        if "_" in question_id:

            img_id = question_id.rsplit('_', 1)[0]
        else:
            img_id = question_id


        image_path = item.get('image', 'unknown.jpg')
        if 'category' in item:
            category = item['category']
        elif '/' in image_path:
            category = image_path.split('/')[-2]
        else:
            category = 'Overall'

        results_by_category[category][img_id].append(is_correct)

    total_acc_sum = 0
    total_acc_plus_sum = 0

    print(f"{'Category':<20} | {'Accuracy':<10} | {'Accuracy+':<10} | {'Score (Max 200)':<10}")
    print("-" * 60)

    for category, images in results_by_category.items():
        total_questions = 0
        correct_questions = 0

        total_images = 0
        correct_images = 0

        for img_id, outcomes in images.items():
            total_images += 1
            total_questions += len(outcomes)
            correct_questions += sum(outcomes)


            if len(outcomes) == 2 and sum(outcomes) == 2:
                correct_images += 1

        acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
        acc_plus = (correct_images / total_images) * 100 if total_images > 0 else 0
        score = acc + acc_plus

        total_acc_sum += acc
        total_acc_plus_sum += acc_plus

        print(f"{category:<20} | {acc:<10.2f} | {acc_plus:<10.2f} | {score:<10.2f}")

    print("-" * 60)

    num_categories = len(results_by_category)
    if num_categories > 0:
        avg_acc = total_acc_sum / num_categories
        avg_acc_plus = total_acc_plus_sum / num_categories
        total_score = total_acc_sum + total_acc_plus_sum
        print(f"{'TOTAL / AVERAGE':<20} | {avg_acc:<10.2f} | {avg_acc_plus:<10.2f} | {total_score:<10.2f}")

Evaluating MME Benchmark for: /content/MME/mme_llava_v1_5_7b_answers_var_a.jsonl

--------------------------------------------------
Category             | Accuracy   | Accuracy+  | Score (Max 200)
------------------------------------------------------------
Overall              | 74.00      | 51.00      | 125.00    
------------------------------------------------------------
TOTAL / AVERAGE      | 74.00      | 51.00      | 125.00    


In [ ]:
# Evaluate POPE Results

result_file = "/content/POPE/pope_llava_v1_5_7b_answers_var_a.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the POPE jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"✅ Found file! Evaluating: {result_file}\n")
    print("-" * 50)

    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    pred_list = []
    label_list = []

    for item in answers:

        gt = str(item.get('label', '')).lower().strip()
        label_list.append(0 if gt == 'no' else 1)


        text = str(item.get('response', '')).strip()


        if '.' in text:
            text = text.split('.')[0]

        text = text.replace(',', '')
        words = text.split(' ')


        if 'No' in words or 'not' in words or 'no' in words:
            pred_list.append(0)
        else:
            pred_list.append(1)

    pos, neg = 1, 0
    yes_ratio = pred_list.count(1) / len(pred_list) if pred_list else 0

    TP, TN, FP, FN = 0, 0, 0, 0
    for pred, label in zip(pred_list, label_list):
        if pred == pos and label == pos:
            TP += 1
        elif pred == pos and label == neg:
            FP += 1
        elif pred == neg and label == neg:
            TN += 1
        elif pred == neg and label == pos:
            FN += 1

    print('TP\tFP\tTN\tFN')
    print(f'{TP}\t{FP}\t{TN}\t{FN}\n')

    precision = float(TP) / float(TP + FP) if (TP + FP) > 0 else 0
    recall = float(TP) / float(TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1 score:  {f1:.4f}')
    print(f'Yes ratio: {yes_ratio:.4f}')
    print("-" * 50)

✅ Found file! Evaluating: /content/POPE/pope_llava_v1_5_7b_answers_var_a.jsonl

--------------------------------------------------
TP	FP	TN	FN
34	7	33	6

Accuracy:  0.8375
Precision: 0.8293
Recall:    0.8500
F1 score:  0.8395
Yes ratio: 0.5125
--------------------------------------------------


In [ ]:
# Var B
import torch.nn as nn



class TargetedVARLayerWrapper(nn.Module):
    def __init__(self, original_layer, target_heads, sink_indices):
        super().__init__()
        self.original_layer = original_layer
        self.target_heads = target_heads
        self.sink_indices = sink_indices

    def forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None, output_attentions=False, use_cache=False, cache_position=None, **kwargs):
        if attention_mask is not None and len(attention_mask.shape) == 4:
            bsz, heads, q_len, kv_len = attention_mask.shape
            num_heads = getattr(self.original_layer.self_attn, "num_heads", heads)

            if heads == 1:
                expanded_mask = attention_mask.expand(bsz, num_heads, q_len, kv_len).clone()
            else:
                expanded_mask = attention_mask.clone()

            min_dtype = torch.finfo(hidden_states.dtype).min
            valid_sinks = [idx for idx in self.sink_indices if idx < kv_len]

            if valid_sinks:
                for head in self.target_heads:
                    expanded_mask[:, head, :, valid_sinks] = min_dtype

            modified_mask = expanded_mask
        else:
            modified_mask = attention_mask

        return self.original_layer(
            hidden_states,
            attention_mask=modified_mask,
            position_ids=position_ids,
            past_key_value=past_key_value,
            output_attentions=output_attentions,
            use_cache=use_cache,
            cache_position=cache_position,
            **kwargs
        )

def generate_benchmark_answers_variant_b(benchmark_dir, question_filename, output_filename, model_id_name="llava-v1.5-7b", drop_k=50, target_layer_count=4, target_head_ratio=0.5):
    input_jsonl_path = os.path.join(benchmark_dir, "Questions", question_filename)
    output_jsonl_path = os.path.join(benchmark_dir, output_filename)
    images_dir = os.path.join(benchmark_dir, "Images")

    if not os.path.exists(input_jsonl_path):
        print(f"Warning: {input_jsonl_path} not found. Skipping...")
        return

    data = []
    with open(input_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))

    print(f"Loaded {len(data)} questions from {input_jsonl_path}")


    IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<image>")
    if IMAGE_TOKEN_ID is None:
        IMAGE_TOKEN_ID = processor.tokenizer.pad_token_id

    def get_token_ids(word_list):
        raw_ids = processor.tokenizer(word_list, add_special_tokens=False).input_ids
        return set(item for sublist in raw_ids for item in (sublist if isinstance(sublist, list) else [sublist]))

    yes_ids = get_token_ids(["Yes", " Yes", "yes", " yes"])
    no_ids = get_token_ids(["No", " No", "no", " no"])
    a_ids = get_token_ids(["(a)", " (a)", "(a", " (", "(", "a", " a", "A", " A"])
    b_ids = get_token_ids(["(b)", " (b)", "(b", "b", " b", "B", " B"])

    NUM_IMAGE_TOKENS = 576
    MAX_NEW_TOKENS = 50

    transformer_layers = None
    for name, module in model.named_modules():
        if isinstance(module, nn.ModuleList) and len(module) > 0 and hasattr(module[0], 'self_attn'):
            transformer_layers = module
            break

    if transformer_layers is None:
        raise ValueError("Could not locate transformer layers.")

    total_layers = len(transformer_layers)
    num_heads = getattr(transformer_layers[0].self_attn, "num_heads", 32)
    target_heads = list(range(int(num_heads * target_head_ratio)))
    target_layers_indices = list(range(total_layers - target_layer_count, total_layers))

    with open(output_jsonl_path, 'w', encoding='utf-8') as outfile:
        for item in tqdm(data, desc=f"Processing {os.path.basename(benchmark_dir)}"):
            image_filename = item.get("image", "")
            prompt_text = item.get("text", item.get("question", "")).strip()
            if not prompt_text:
                continue

            img_path = os.path.join(images_dir, image_filename)
            if not os.path.exists(img_path):
                img_path = os.path.join(benchmark_dir, image_filename)

            try:
                raw_image = Image.open(img_path).convert("RGB")
            except Exception as e:
                continue

            full_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"

            inputs = processor(text=full_prompt, images=raw_image, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.inference_mode():
                # Time
                start_time = time.perf_counter()


                outputs_base = model(**inputs, use_cache=True, output_hidden_states=False)
                past_key_values = outputs_base.past_key_values

                v_caches = []
                cache_iterable = past_key_values.to_legacy_cache() if hasattr(past_key_values, "to_legacy_cache") else (
                    list(zip(past_key_values.key_cache, past_key_values.value_cache)) if hasattr(past_key_values, "key_cache") else past_key_values)

                for layer_data in cache_iterable:
                    v_caches.append(layer_data[1])

                seq_len = v_caches[0].shape[2]

                input_ids_list = inputs["input_ids"][0].tolist()
                image_token_index = 32000
                vis_start = input_ids_list.index(image_token_index) if image_token_index in input_ids_list else (
                    input_ids_list.index(IMAGE_TOKEN_ID) if IMAGE_TOKEN_ID in input_ids_list else -1)

                global_sink_indices = []
                if vis_start != -1:
                    vis_end = vis_start + NUM_IMAGE_TOKENS
                    v_norms = []
                    for v_cache in v_caches:
                        vis_v_cache = v_cache[0, :, vis_start:vis_end, :]
                        layer_v_norm = torch.linalg.norm(vis_v_cache.float(), dim=-1).mean(dim=0)
                        v_norms.append(layer_v_norm)

                    avg_v_norm = torch.stack(v_norms).mean(dim=0)
                    _, topk_indices = torch.topk(avg_v_norm, k=drop_k)
                    global_sink_indices = (topk_indices + vis_start).tolist()

                # Wrapper
                original_layer_refs = {}
                for idx in target_layers_indices:
                    original_layer_refs[idx] = transformer_layers[idx]
                    transformer_layers[idx] = TargetedVARLayerWrapper(
                        original_layer=original_layer_refs[idx],
                        target_heads=target_heads,
                        sink_indices=global_sink_indices
                    )

                try:
                    # Prefill
                    outputs_var = model(**inputs, use_cache=True, output_hidden_states=False)
                    first_token_logits = outputs_var.logits[0, -1, :]
                    current_past_key_values = outputs_var.past_key_values

                    # Prob. Logic
                    probs = F.softmax(first_token_logits.float(), dim=-1)

                    is_multiple_choice = "(a)" in prompt_text.lower() and "(b)" in prompt_text.lower()

                    if is_multiple_choice:
                        opt1_prob = sum(probs[idx].item() for idx in a_ids if idx < probs.size(-1))
                        opt2_prob = sum(probs[idx].item() for idx in b_ids if idx < probs.size(-1))
                    else:
                        opt1_prob = sum(probs[idx].item() for idx in yes_ids if idx < probs.size(-1))
                        opt2_prob = sum(probs[idx].item() for idx in no_ids if idx < probs.size(-1))

                    top_k_probs, top_k_ids = torch.topk(probs, k=5)
                    top_k_tokens = processor.tokenizer.convert_ids_to_tokens(top_k_ids)
                    top_k_dict = {token: round(prob.item(), 4) for token, prob in zip(top_k_tokens, top_k_probs)}
                    entropy = -(probs * torch.log(probs + 1e-9)).sum().item()

                    # Decode loop
                    next_token_id = first_token_logits.argmax(dim=-1).unsqueeze(0).unsqueeze(0)
                    generated_ids = [next_token_id.item()]
                    current_input_ids = next_token_id

                    current_attention_mask = torch.ones((1, seq_len + 1), device=model.device, dtype=torch.long)
                    current_attention_mask[0, :inputs["attention_mask"].shape[1]] = inputs["attention_mask"][0]

                    for _ in range(MAX_NEW_TOKENS - 1):
                        outputs = model(
                            input_ids=current_input_ids,
                            attention_mask=current_attention_mask,
                            past_key_values=current_past_key_values,
                            use_cache=True
                        )
                        current_past_key_values = outputs.past_key_values
                        next_token_id = outputs.logits[0, -1, :].argmax(dim=-1).unsqueeze(0).unsqueeze(0)

                        token_val = next_token_id.item()
                        generated_ids.append(token_val)
                        if token_val == processor.tokenizer.eos_token_id:
                            break

                        current_input_ids = next_token_id
                        current_attention_mask = torch.cat(
                            [current_attention_mask, torch.ones((1, 1), device=model.device, dtype=torch.long)], dim=1
                        )
                finally:
                    for idx in target_layers_indices:
                        transformer_layers[idx] = original_layer_refs[idx]


                end_time = time.perf_counter()
                inference_time = end_time - start_time

            final_output_ids = torch.tensor([generated_ids], device=model.device)
            response_text = processor.decode(final_output_ids[0], skip_special_tokens=True).strip()

            output_item = {
                "question_id": item.get("question_id", ""),
                "prompt": prompt_text,
                "label": item.get("label", ""),
                "response": response_text,
                "image": image_filename,
                "model_id": f"{model_id_name}_varB",
                "opt1_prob": round(opt1_prob, 4),
                "opt2_prob": round(opt2_prob, 4),
                "entropy": round(entropy, 4),
                "top_5_distribution": top_k_dict,
                "inference_time": round(inference_time, 4)
            }
            outfile.write(json.dumps(output_item) + "\n")



In [ ]:
benchmarks = [
    {
        "name": "MME",
        "path": "/content/MME",
        "questions_file": "mme-questions.jsonl"
    },
    {
        "name": "MMVP",
        "path": "/content/MMVP",
        "questions_file": "mmvp-questions.jsonl"
    },
    {
        "name": "POPE",
        "path": "/content/POPE",
        "questions_file": "pope-questions.jsonl"
    }
]

for bm in benchmarks:


    output_filename = f"{bm['name'].lower()}_llava_v1_5_7b_answers_var_b.jsonl"

    generate_benchmark_answers_variant_b(
        benchmark_dir=bm["path"],
        question_filename=bm["questions_file"],
        output_filename=output_filename,
        model_id_name="llava-v1.5-7b"
    )



Loaded 200 questions from /content/MME/Questions/mme-questions.jsonl


Processing MME: 100%|██████████| 200/200 [02:25<00:00,  1.37it/s]


Loaded 300 questions from /content/MMVP/Questions/mmvp-questions.jsonl


Processing MMVP: 100%|██████████| 300/300 [05:28<00:00,  1.09s/it]


Loaded 80 questions from /content/POPE/Questions/pope-questions.jsonl


Processing POPE: 100%|██████████| 80/80 [02:53<00:00,  2.17s/it]


In [ ]:
# Evaluate MME Results

result_file = "/content/MME/mme_llava_v1_5_7b_answers_var_b.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the MME jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"Evaluating MME Benchmark for: {result_file}\n")
    print("-" * 50)


    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    results_by_category = defaultdict(lambda: defaultdict(list))

    for item in answers:
        gt = str(item.get('label', '')).lower().strip()
        text = str(item.get('response', '')).lower().strip()


        if '.' in text:
            text = text.split('.')[0]
        text = text.replace(',', '').split(' ')

        if 'no' in text or 'not' in text:
            pred = 'no'
        else:
            pred = 'yes'

        is_correct = (pred == gt)


        question_id = item.get("question_id", "")
        if "_" in question_id:

            img_id = question_id.rsplit('_', 1)[0]
        else:
            img_id = question_id


        image_path = item.get('image', 'unknown.jpg')
        if 'category' in item:
            category = item['category']
        elif '/' in image_path:
            category = image_path.split('/')[-2]
        else:
            category = 'Overall'

        results_by_category[category][img_id].append(is_correct)

    total_acc_sum = 0
    total_acc_plus_sum = 0

    print(f"{'Category':<20} | {'Accuracy':<10} | {'Accuracy+':<10} | {'Score (Max 200)':<10}")
    print("-" * 60)

    for category, images in results_by_category.items():
        total_questions = 0
        correct_questions = 0

        total_images = 0
        correct_images = 0

        for img_id, outcomes in images.items():
            total_images += 1
            total_questions += len(outcomes)
            correct_questions += sum(outcomes)


            if len(outcomes) == 2 and sum(outcomes) == 2:
                correct_images += 1

        acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
        acc_plus = (correct_images / total_images) * 100 if total_images > 0 else 0
        score = acc + acc_plus

        total_acc_sum += acc
        total_acc_plus_sum += acc_plus

        print(f"{category:<20} | {acc:<10.2f} | {acc_plus:<10.2f} | {score:<10.2f}")

    print("-" * 60)

    num_categories = len(results_by_category)
    if num_categories > 0:
        avg_acc = total_acc_sum / num_categories
        avg_acc_plus = total_acc_plus_sum / num_categories
        total_score = total_acc_sum + total_acc_plus_sum
        print(f"{'TOTAL / AVERAGE':<20} | {avg_acc:<10.2f} | {avg_acc_plus:<10.2f} | {total_score:<10.2f}")

Evaluating MME Benchmark for: /content/MME/mme_llava_v1_5_7b_answers_var_b.jsonl

--------------------------------------------------
Category             | Accuracy   | Accuracy+  | Score (Max 200)
------------------------------------------------------------
Overall              | 73.00      | 48.00      | 121.00    
------------------------------------------------------------
TOTAL / AVERAGE      | 73.00      | 48.00      | 121.00    


In [ ]:
# Evaluate POPE Results

result_file = "/content/POPE/pope_llava_v1_5_7b_answers_var_b.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the POPE jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"✅ Found file! Evaluating: {result_file}\n")
    print("-" * 50)

    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    pred_list = []
    label_list = []

    for item in answers:

        gt = str(item.get('label', '')).lower().strip()
        label_list.append(0 if gt == 'no' else 1)


        text = str(item.get('response', '')).strip()


        if '.' in text:
            text = text.split('.')[0]

        text = text.replace(',', '')
        words = text.split(' ')


        if 'No' in words or 'not' in words or 'no' in words:
            pred_list.append(0)
        else:
            pred_list.append(1)

    pos, neg = 1, 0
    yes_ratio = pred_list.count(1) / len(pred_list) if pred_list else 0

    TP, TN, FP, FN = 0, 0, 0, 0
    for pred, label in zip(pred_list, label_list):
        if pred == pos and label == pos:
            TP += 1
        elif pred == pos and label == neg:
            FP += 1
        elif pred == neg and label == neg:
            TN += 1
        elif pred == neg and label == pos:
            FN += 1

    print('TP\tFP\tTN\tFN')
    print(f'{TP}\t{FP}\t{TN}\t{FN}\n')

    precision = float(TP) / float(TP + FP) if (TP + FP) > 0 else 0
    recall = float(TP) / float(TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1 score:  {f1:.4f}')
    print(f'Yes ratio: {yes_ratio:.4f}')
    print("-" * 50)

✅ Found file! Evaluating: /content/POPE/pope_llava_v1_5_7b_answers_var_b.jsonl

--------------------------------------------------
TP	FP	TN	FN
34	6	34	6

Accuracy:  0.8500
Precision: 0.8500
Recall:    0.8500
F1 score:  0.8500
Yes ratio: 0.5000
--------------------------------------------------


In [ ]:
import re

result_files = {
    "Variant A": "/content/MMVP/mmvp_llava_v1_5_7b_answers_var_a.jsonl",
    "Variant B": "/content/MMVP/mmvp_llava_v1_5_7b_answers_var_b.jsonl",
    "Variant C": "/content/MMVP/mmvp_llava_v1_5_7b_answers_c.jsonl"
}

def evaluate_mmvp_file(filepath, variant_name):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}. Skipping {variant_name}...\n")
        return

    with open(filepath, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f if line.strip()]

    total_questions = len(data)
    correct_questions = 0


    pairs_tracker = {}

    for item in data:
        question_id = int(item.get("question_id", 0))
        pair_id = (question_id + 1) // 2

        prompt = item.get("prompt", "").lower()
        label = item.get("label", "").lower().strip() # usually "(a)" or "(b)"
        response = item.get("response", "").lower().strip()


        option_a_text = ""
        option_b_text = ""

        match_a = re.search(r'\(a\)\s*(.*?)\s*\(\s*b\s*\)', prompt)
        match_b = re.search(r'\(\s*b\s*\)\s*(.*)', prompt)

        if match_a:
            option_a_text = match_a.group(1).strip()
        if match_b:
            option_b_text = match_b.group(1).strip()


        option_a_text_clean = re.sub(r'[^\w\s]', '', option_a_text)
        option_b_text_clean = re.sub(r'[^\w\s]', '', option_b_text)
        response_clean = re.sub(r'[^\w\s]', '', response)


        predicted_label = None


        if response.startswith("(a)") or response.startswith("a)") or response.startswith("a."):
            predicted_label = "(a)"
        elif response.startswith("(b)") or response.startswith("b)") or response.startswith("b."):
            predicted_label = "(b)"
        else:

            has_a = option_a_text_clean in response_clean if option_a_text_clean else False
            has_b = option_b_text_clean in response_clean if option_b_text_clean else False

            if has_a and not has_b:
                predicted_label = "(a)"
            elif has_b and not has_a:
                predicted_label = "(b)"
            else:

                opt1_prob = item.get("opt1_prob", 0.0)
                opt2_prob = item.get("opt2_prob", 0.0)
                predicted_label = "(a)" if opt1_prob > opt2_prob else "(b)"


        is_correct = (predicted_label == label)
        if is_correct:
            correct_questions += 1


        if pair_id not in pairs_tracker:
            pairs_tracker[pair_id] = []
        pairs_tracker[pair_id].append(is_correct)


    correct_pairs = 0
    total_pairs = len(pairs_tracker)

    for p_id, outcomes in pairs_tracker.items():

        if len(outcomes) == 2 and all(outcomes):
            correct_pairs += 1

    q_acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
    p_acc = (correct_pairs / total_pairs) * 100 if total_pairs > 0 else 0

    print(f"=== {variant_name} ===")
    print(f"Question-Level Accuracy: {correct_questions}/{total_questions} ({q_acc:.2f}%)")
    print(f"Pair-Level Accuracy:     {correct_pairs}/{total_pairs} ({p_acc:.2f}%)\n")



print("Evaluating MMVP Benchmarks...\n" + "="*40 + "\n")
for variant, path in result_files.items():
    evaluate_mmvp_file(path, variant)

Evaluating MMVP Benchmarks...

=== Variant A ===
Question-Level Accuracy: 179/300 (59.67%)
Pair-Level Accuracy:     33/150 (22.00%)

=== Variant B ===
Question-Level Accuracy: 174/300 (58.00%)
Pair-Level Accuracy:     32/150 (21.33%)

=== Variant C ===
Question-Level Accuracy: 180/300 (60.00%)
Pair-Level Accuracy:     35/150 (23.33%)

